# Step 3: Multi-Scholar Semantic Search & AI Synthesis

This notebook provides a complete, runnable end-to-end pipeline for the upgraded **Multi-Scholar Islamic RAG System**.

### Key Capabilities:
1. **Multi-Scholar Dataset Support**: Indexes fatwas across major Sunni/Salafi scholars and official bodies:
   - **Sheikh Abdulaziz ibn Baz**
   - **Sheikh Muhammad ibn Saalih al-Uthaymeen**
   - **Sheikh Saleh Al-Fawzan**
   - **Al-Lajnah Ad-Da'imah lil-Ifta** (Permanent Committee / Council of Senior Scholars)
2. **Two Search Modes**:
   - **Mode 1**: **X answers per scholar** (searches and reranks each scholar's corpus independently to guarantee balanced representation).
   - **Mode 2**: **X answers ranked overall** (searches and reranks across all selected scholars as a unified pool).
3. **Two-Stage Neural Retrieval**: Precomputed `Qwen3-Embedding-0.6B` vectors + `ALJIACHI/Mizan-Rerank-V2` cross-encoder.
4. **AI Review & Judging**: Per-fatwa grounded verdict (`answers-it`, `related-only`, `unrelated`).
5. **New AI Synthesis Features**:
   - **Consensus Classification** (`agree` / `partially-agree` / `disagree`) evaluated using **Gemini** or **JEV** (`typesafe/jev-*`).
   - **Comparative Summary**: A coherent synthesis of what the scholars say and rule, generated via **Gemini**.

In [ ]:
# 1. Setup & Environment
!pip install -q sentence-transformers transformers google-genai torch pandas ftfy

import os, sys, json, time, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer, CrossEncoder

# Load environment variables from .env if present
for env_path in [Path('.env'), Path('../.env')]:
    if env_path.exists():
        with open(env_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    k, v = line.split('=', 1)
                    os.environ.setdefault(k.strip(), v.strip().strip('"\''))

gemini_ok = bool(os.getenv('GEMINI_API_KEY'))
openrouter_ok = bool(os.getenv('OPENROUTER_API_KEY'))
print(f'Gemini API Key:     {"✓ Set" if gemini_ok else "✕ Missing (set GEMINI_API_KEY)"}')
print(f'OpenRouter API Key: {"✓ Set" if openrouter_ok else "✕ Missing (set OPENROUTER_API_KEY)"}')
print(f'CUDA available:     {torch.cuda.is_available()}')

In [ ]:
# 2. Load Multi-Scholar Assets (with Graceful Fallback)
DATA_DIRS = [Path('data'), Path('../data'), Path('data/cleaned'), Path('../data/cleaned')]

def find_file(filename: str) -> Path | None:
    for d in DATA_DIRS:
        p = d / filename
        if p.exists():
            return p
    return None

# Look for multi-scholar artifacts, or fall back to Bin Baz if Kaggle Step 3 has not run yet
multi_csv = find_file('multi_scholar_clean.csv')
multi_emb = find_file('qwen3_06b_multi_scholar_embeddings.pt')
binbaz_csv = find_file('binbaz_clean.csv')
binbaz_emb = find_file('qwen3_06b_title_answer_embeddings.pt')

if multi_csv and multi_emb:
    print(f'Loading Multi-Scholar Corpus: {multi_csv}')
    df = pd.read_csv(multi_csv)
    embeddings = torch.load(multi_emb, map_location='cpu', weights_only=True)
elif binbaz_csv and binbaz_emb:
    print('Multi-scholar embeddings not detected yet — loading existing Bin Baz corpus as base.')
    df = pd.read_csv(binbaz_csv)
    embeddings = torch.load(binbaz_emb, map_location='cpu', weights_only=True)
else:
    raise FileNotFoundError('No valid CSV and embeddings pair found in data directories!')

if isinstance(embeddings, dict):
    for k in ('embeddings', 'tensor', 'data'):
        if k in embeddings:
            embeddings = embeddings[k]
            break
embeddings = F.normalize(torch.as_tensor(embeddings, dtype=torch.float32), p=2, dim=1)
df = df.iloc[:len(embeddings)].copy().reset_index(drop=True)
print(f'Indexed {len(df):,} fatwas · Embeddings matrix shape: {tuple(embeddings.shape)}')
print('\nScholars available in index:')
print(df['source'].value_counts())

In [ ]:
# 3. Load Retrieval & Reranker Models
device_embedder = 'cuda' if torch.cuda.is_available() else 'cpu'
device_reranker = 'cuda' if torch.cuda.is_available() else 'cpu'

EMBEDDER_MODEL = 'Qwen/Qwen3-Embedding-0.6B'
RERANKER_MODEL = 'ALJIACHI/Mizan-Rerank-V2'

print(f'Loading embedder ({EMBEDDER_MODEL}) on {device_embedder}...')
embedder = SentenceTransformer(EMBEDDER_MODEL, device=device_embedder)
embedder.max_seq_length = 1024

print(f'Loading reranker ({RERANKER_MODEL}) on {device_reranker}...')
reranker = CrossEncoder(RERANKER_MODEL, max_length=1024, device=device_reranker, trust_remote_code=True)
print('Models ready.')

In [ ]:
# 4. Search Engines: Mode 1 and Mode 2

def encode_query(query: str) -> torch.Tensor:
    vec = embedder.encode(query, convert_to_tensor=True, device=device_embedder, normalize_embeddings=True)
    return vec.detach().cpu().float()

def rerank(query: str, candidates: pd.DataFrame) -> pd.DataFrame:
    if len(candidates) == 0:
        return candidates
    pairs = [[query, str(t)] for t in candidates['embed_text'].tolist()]
    scores = np.asarray(reranker.predict(pairs, batch_size=16, show_progress_bar=False)).reshape(-1)
    res = candidates.copy()
    res['reranker_score'] = scores
    res = res.sort_values('reranker_score', ascending=False).reset_index(drop=True)
    res['rerank_rank'] = np.arange(1, len(res) + 1)
    return res

# MODE 1: X answers per selected scholar (each scholar dataset searched independently)
def search_mode_per_scholar(query: str, scholars: list[str], x_per_scholar: int = 2, pool_per_scholar: int = 10) -> pd.DataFrame:
    q_vec = encode_query(query)
    results = []
    for scholar in scholars:
        scholar_mask = (df['source'] == scholar).values
        scholar_indices = np.where(scholar_mask)[0]
        if len(scholar_indices) == 0:
            continue
        sub_emb = embeddings[scholar_indices]
        sims = torch.mv(sub_emb, q_vec)
        k_cand = min(pool_per_scholar, len(scholar_indices))
        top_vals, top_rel_idx = torch.topk(sims, k=k_cand)
        top_abs_idx = scholar_indices[top_rel_idx.numpy()]
        cand_df = df.iloc[top_abs_idx].copy().reset_index(drop=True)
        cand_df['retrieval_rank'] = np.arange(1, len(cand_df) + 1)
        cand_df['embedding_score'] = top_vals.numpy()
        reranked = rerank(query, cand_df)
        results.append(reranked.head(x_per_scholar))
    if results:
        out = pd.concat(results, ignore_index=True)
        out['global_rank'] = np.arange(1, len(out) + 1)
        return out
    return pd.DataFrame()

# MODE 2: X answers ranked overall (across all selected scholars)
def search_mode_overall(query: str, scholars: list[str], x_total: int = 5, pool_total: int = 20) -> pd.DataFrame:
    q_vec = encode_query(query)
    scholar_mask = df['source'].isin(scholars).values
    indices = np.where(scholar_mask)[0]
    if len(indices) == 0:
        return pd.DataFrame()
    sub_emb = embeddings[indices]
    sims = torch.mv(sub_emb, q_vec)
    k_cand = min(pool_total, len(indices))
    top_vals, top_rel_idx = torch.topk(sims, k=k_cand)
    top_abs_idx = indices[top_rel_idx.numpy()]
    cand_df = df.iloc[top_abs_idx].copy().reset_index(drop=True)
    cand_df['retrieval_rank'] = np.arange(1, len(cand_df) + 1)
    cand_df['embedding_score'] = top_vals.numpy()
    reranked = rerank(query, cand_df)
    return reranked.head(x_total).reset_index(drop=True)

In [ ]:
# 5. AI Synthesis: Consensus Classification & Comparative Summary

def format_fatwas_for_llm(candidates: pd.DataFrame, max_chars: int = 1500) -> str:
    lines = []
    for i, row in candidates.iterrows():
        lines.append(f"--- Fatwa [{i+1}] by {row['source']} ---")
        lines.append(f"Title: {row['title']}")
        lines.append(f"Question: {str(row['question'])[:300]}")
        lines.append(f"Answer: {str(row['answer'])[:max_chars]}\n")
    return '\n'.join(lines)

def classify_consensus_gemini(user_query: str, candidates: pd.DataFrame) -> dict:
    """Determines whether the retrieved scholars agree, partially agree, or disagree."""
    if not os.getenv('GEMINI_API_KEY'):
        return {'status': 'skipped', 'reason': 'GEMINI_API_KEY not set'}
    from google import genai
    client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
    prompt = (
        f"User Question: {user_query}\n\n"
        f"Scholarly Fatwas:\n{format_fatwas_for_llm(candidates)}\n\n"
        "Task: Compare the rulings and positions of the scholars regarding the user question.\n"
        "Determine their level of agreement on the core ruling into one of three classes:\n"
        "- 'agree': All scholars share the same or harmonized ruling/stance.\n"
        "- 'partially-agree': General agreement on the fundamental ruling, with slight divergence in details, conditions, or exceptions.\n"
        "- 'disagree': Evident disagreement or differing rulings (e.g., permissible vs prohibited, obligatory vs recommended).\n\n"
        "Return a valid JSON object with keys: 'consensus' ('agree'|'partially-agree'|'disagree'), 'confidence' (float 0.0-1.0), and 'explanation' (Arabic explanation of consensus)."
    )
    resp = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config={'response_mime_type': 'application/json'}
    )
    return json.loads(resp.text)

def summarize_positions_gemini(user_query: str, candidates: pd.DataFrame) -> str:
    """Produces an objective Arabic summary synthesizing the scholarly viewpoints."""
    if not os.getenv('GEMINI_API_KEY'):
        return 'Gemini API key not configured.'
    from google import genai
    client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
    prompt = (
        f"السؤال: {user_query}\n\n"
        f"فتاوى العلماء:\n{format_fatwas_for_llm(candidates)}\n\n"
        "المطلوب: قدم ملخصاً فقهياً جامعاً وواضحاً باللغة العربية يوضح موقف العلماء وأقوالهم بشأن هذه المسألة، "
        "مع نسبة كل قول إلى الشيخ أو الهيئة التي أفتت به بإيجاز ودقة وبدون تحيز."
    )
    resp = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    return resp.text.strip()

In [ ]:
# 6. Execution & Demo: Mode 1 (X answers per Scholar)
test_query = 'ما هي كفارة اليمين ومتى يشرع الإطعام ومتى يصام؟'
available_scholars = df['source'].unique().tolist()
print(f'Query: {test_query}\nAvailable scholars: {available_scholars}\n')

print('=== EXECUTING SEARCH MODE 1 (2 answers per scholar) ===')
results_mode1 = search_mode_per_scholar(test_query, available_scholars, x_per_scholar=2)
for i, r in results_mode1.iterrows():
    print(f"[{r['source']}] Fatwa #{r['fatwa_id']}: {r['title']} (Rerank score: {r['reranker_score']:.3f})")

In [ ]:
# 7. Execution & Demo: Mode 2 (Top 5 Overall)
print('=== EXECUTING SEARCH MODE 2 (Top 5 Overall across scholars) ===')
results_mode2 = search_mode_overall(test_query, available_scholars, x_total=5)
for i, r in results_mode2.iterrows():
    print(f"#{i+1} [{r['source']}] {r['title']} (Rerank score: {r['reranker_score']:.3f})")

In [ ]:
# 8. Demo: AI Consensus & Summary
if gemini_ok and len(results_mode1) > 0:
    print('Classifying scholarly consensus...')
    consensus = classify_consensus_gemini(test_query, results_mode1)
    print('Consensus Result:', json.dumps(consensus, ensure_ascii=False, indent=2))
    
    print('\nGenerating Scholarly Summary...')
    summary = summarize_positions_gemini(test_query, results_mode1)
    print('\n--- Scholarly Summary ---')
    print(summary)
else:
    print('Set GEMINI_API_KEY in .env to test real-time AI consensus classification and summary.')